<a href="https://colab.research.google.com/github/Turbostar123/AI-Item-Scanner/blob/main/ShoppingList.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This AI program will allow you to scan an image using AI. The AI will list out the items it sees, as well as the quantities of each item.

I will also be improving this code by...
1. Allowing the AI to predict the price of each item.
2. Calculating the total cost of the items, along with the total cost after taxes.
3. Giving the program the ability to print out your results, like a reciept, and download it as a pdf.




Before we start we need to install the appropriate libraries and assign our API key:

In [1]:
!pip install -U google-genai
!pip install reportlab

import google.genai as genai
from google.colab import userdata
from google.colab import files

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

genModel = 'gemini-2.5-flash-lite'

First we will allow the AI to scan an image and list the items scanned, as well as their quantities.

Example Photo: https://medium.com/@ianocampo95/the-importance-of-grocery-shopping-dcbda019139f

When using a photo, upload to colab and rename the image to "ShoppingCartImage". Must be a jpeg.

You can also use the following cell to download the sample image instead of using your own.

In [2]:
import requests
import os

new_image_name = "ShoppingCartImage.jpg"
image_url = "https://miro.medium.com/v2/resize:fit:1400/format:webp/1*mo0myR6C7krWTipKq_rJ4A.jpeg"

response = requests.get(image_url)
if response.status_code == 200:
    with open(new_image_name, 'wb') as f:
        f.write(response.content)
    print(f"Image downloaded and saved as '{new_image_name}'")
else:
    print(f"Failed to download image from {image_url}. Status code: {response.status_code}")

Image downloaded and saved as 'ShoppingCartImage.jpg'


Now that the image is downloaded in colab and renamed, we can use the AI to scan the image.

In [3]:
image = open("ShoppingCartImage.jpg", "rb").read() #Assigns 'image' as the preuploaded image. Must use the file name of the image uploaded to colab.
ai_list_output = "" #Used to store generated list of items for future manipulation

for itemList in client.models.generate_content_stream( #Lets the AI model scan the image and retreive chunks of information that can be outputted as text
    model=genModel,
    contents=['Scan the image. List the items shown only in the shopping cart and the quantity of each item. When listing quantity, only use integers. The format should look like this: * {Item}: {Quantity}',
              genai.types.Part.from_bytes(data=image, mime_type="image/jpeg")
              ]
):
  print(itemList.text, end='')
  ai_list_output += itemList.text

* Boneless Skinless Chicken Breasts: 1
* Bananas: 1
* Oranges: 1
* Asparagus: 1
* Tomatoes: 1
* Ground Beef: 1
* Bread: 2
* Mustard: 1
* Pickles: 1
* Cheese Heads: 1
* Eggs: 2
* Milk: 1
* Cream Cheese: 1

**Improvement #1:** We want to take this list of items and assign a price to them. We also want to get the total cost of each item based on the quantity.

In [4]:
#First we will make a dictionary for the AI to use to predict the prices of each item. This will be useful later as well.
print(ai_list_output, end="\n\n") #Prints the list of items again

item_dict = {}  # dictionary to store items

lines = ai_list_output.splitlines()
for line in lines:
    line = line.strip()
    if line.startswith("*"):  # only process lines that start with "*"
        parts = line.lstrip("* ").split(":") # Remove "* " and split by ":" to separate item name and quantity
        if len(parts) == 2: #Check for two parts (Item and integer value)
            name = parts[0].strip() #Assign item to name
            try:
                qty = int(parts[1].strip()) #Assign int value to quantity
            except ValueError:
                # Skip lines with non-integer quantities
                continue
            # Add to dictionary only if not already present
            if name not in item_dict:
                item_dict[name] = qty

#print(item_dict, end="\n\n") #Can be used to see what the dictionary looks like at this point

#-------------------------------------------------------------------------------

#Secondly, we will let the AI predict the prices of each item and output the results in a list
ai_price_output = "" #Used to store generated list of prices for each item

for prices in client.models.generate_content_stream( #Estimates price using AI. The AI will use the dictionary we just made
    model = genModel,
    contents = ['Look at this list of items. Give an estimate on the price of each **INDIVIDUAL** item on the list. Do not output your reason as to why, instead you should strictly follow the format. When outputting the prices for each item, the format should look like this: {Item}: ${Price}', ai_list_output]
):
  ai_price_output += prices.text

print(ai_price_output, end="\n\n") #Prints Prices

#-------------------------------------------------------------------------------

#Finally, we will create a dictionary of dictionaries to neatly store our data
for item, qty in list(item_dict.items()): #Converts our item_dict into a dictionary of dictionaries. Necessary for neatly organizing our data.
        item_dict[item] = {"Quantity": qty}

lines = ai_price_output.splitlines() #Separates each line in the list of prices
for line in lines:
  line = line.strip()
  if line.startswith("*")==True: #Checks to see if line starts with bullet point
    parts = line.lstrip("* ").split(":") #Removes bullet point, splits line between ":"
    if len(parts) == 2: #Checks for two parts
      name = parts[0].strip() #Gets the name, which should be the first part
      try:
        num = parts[1].lstrip(" $") #Takes the dollar sign out
        price = float(num.strip()) #Converts second part into a float value if able
      except ValueError:
        continue
      qty = item_dict[name]["Quantity"]
      item_dict[name]["Price"] = price
      item_dict[name]["Total"] = qty * price

#Now we print out list of items
print("Here's a list of the items and their data:", end="\n\n")
for item in item_dict:
  print("* ", item,":", item_dict[item], end="\n")


* Boneless Skinless Chicken Breasts: 1
* Bananas: 1
* Oranges: 1
* Asparagus: 1
* Tomatoes: 1
* Ground Beef: 1
* Bread: 2
* Mustard: 1
* Pickles: 1
* Cheese Heads: 1
* Eggs: 2
* Milk: 1
* Cream Cheese: 1

* Boneless Skinless Chicken Breasts: $3.50
* Bananas: $0.25
* Oranges: $0.75
* Asparagus: $3.00
* Tomatoes: $0.50
* Ground Beef: $5.00
* Bread: $2.50
* Mustard: $2.00
* Pickles: $3.00
* Cheese Heads: $2.50
* Eggs: $4.00
* Milk: $3.00
* Cream Cheese: $3.00

Here's a list of the items and their data:

*  Boneless Skinless Chicken Breasts : {'Quantity': 1, 'Price': 3.5, 'Total': 3.5}
*  Bananas : {'Quantity': 1, 'Price': 0.25, 'Total': 0.25}
*  Oranges : {'Quantity': 1, 'Price': 0.75, 'Total': 0.75}
*  Asparagus : {'Quantity': 1, 'Price': 3.0, 'Total': 3.0}
*  Tomatoes : {'Quantity': 1, 'Price': 0.5, 'Total': 0.5}
*  Ground Beef : {'Quantity': 1, 'Price': 5.0, 'Total': 5.0}
*  Bread : {'Quantity': 2, 'Price': 2.5, 'Total': 5.0}
*  Mustard : {'Quantity': 1, 'Price': 2.0, 'Total': 2.0}
*  

**Improvement #2:** Now we need to add up the total cost of all the items and multiply by tax rate to get the total cost of the items.

In [5]:
#Printing out the list of items again for reference
print("List of Items:", end="\n\n")
for item in item_dict:
  qty = item_dict[item]["Quantity"]
  price = item_dict[item]["Price"]
  cost = item_dict[item]["Total"]
  print(f"{item}: {qty}, Price Per unit: ${price:.2f}, Total: ${cost:.2f}")
print(end="\n")

item_cost = 0.0
for item in item_dict: #Adds up the total cost of each item
  item_cost += item_dict[item]["Total"]

print(f"Total Cost of Items: ${item_cost:.2f}", end="\n\n") #Prints total cost of all items

tax_rate = 0.08875 #NY State tax rate
total_cost = item_cost+ (tax_rate * item_cost)

print(f"NY State Tax Rate: {tax_rate}","\n"f"Total Cost of Items with Tax: ${total_cost:.2f}", end="\n\n")


List of Items:

Boneless Skinless Chicken Breasts: 1, Price Per unit: $3.50, Total: $3.50
Bananas: 1, Price Per unit: $0.25, Total: $0.25
Oranges: 1, Price Per unit: $0.75, Total: $0.75
Asparagus: 1, Price Per unit: $3.00, Total: $3.00
Tomatoes: 1, Price Per unit: $0.50, Total: $0.50
Ground Beef: 1, Price Per unit: $5.00, Total: $5.00
Bread: 2, Price Per unit: $2.50, Total: $5.00
Mustard: 1, Price Per unit: $2.00, Total: $2.00
Pickles: 1, Price Per unit: $3.00, Total: $3.00
Cheese Heads: 1, Price Per unit: $2.50, Total: $2.50
Eggs: 2, Price Per unit: $4.00, Total: $8.00
Milk: 1, Price Per unit: $3.00, Total: $3.00
Cream Cheese: 1, Price Per unit: $3.00, Total: $3.00

Total Cost of Items: $39.50

NY State Tax Rate: 0.08875 
Total Cost of Items with Tax: $43.01



**Improvement #3:** Now we want to print this information out as a receipt and download it as a pdf.

In [6]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors

monospace = ParagraphStyle(
    'Monospace',
    fontName='Courier',
    fontSize=10,
    leading=12
    )# spacing between lines

# Create PDF
receipt = SimpleDocTemplate("receipt.pdf", pagesize=letter)
styles = getSampleStyleSheet()

elements = []

# Header for receipt
elements.append(Paragraph("STORE RECEIPT", styles["Title"]))
elements.append(Spacer(1, 12))

# Column headers
header = f"{'Item':30} {'Qty':5} {'Price':10} {'Subtotal':10}"
elements.append(Preformatted(header, monospace))
elements.append(Spacer(1, 6))

# Add items to pdf
for item, details in item_dict.items():
    qty = details["Quantity"]
    price = details["Price"]
    subtotal = details["Total"]

    max_item_length = 30
    if len(item) > max_item_length:
      item_display = item[:27] + "..."
    else:
      item_display = item

    line = f"{item_display:30} {qty:<5} ${price:<9.2f} ${subtotal:<9.2f}"
    elements.append(Preformatted(line, monospace))
    elements.append(Spacer(1, 6))

# Subtotal and Total Cost
elements.append(Spacer(1, 12))
elements.append(Paragraph(f"<b>Cost of Items: ${item_cost:.2f}</b>", styles["Heading2"]))
elements.append(Spacer(1, 6))
elements.append(Preformatted(f"Tax Rate: {tax_rate}", monospace))
elements.append(Paragraph(f"<b>Total Cost: ${total_cost:.2f}</b>", styles["Heading2"]))
# Build PDF
receipt.build(elements)

print("PDF generated: receipt.pdf")


PDF generated: receipt.pdf


The receipt should have downloaded and should be shown to the left under the Files tab and should be named `recepit.pdf`.